# 150 — Observabilidad: logs, métricas y trazas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** a) **Métrica** (counter de peticiones; barata de agregar por semana).
b) **Traza**: descompone esa petición concreta en spans y muestra dónde se fue el
tiempo. c) **Log** (evento discreto con stack trace y contexto), idealmente enlazado por
`trace_id`. d) **Métrica** tipo histogram, de la que el backend deriva el p99 contra el
SLO.

**Ejercicio 2.** retrieve 16.7 %, rerank 6.3 %, llm_call **70.8 %**, guardrail 6.3 %.
Se optimiza `llm_call` primero; el atributo clave es `input_tokens=5200`: un contexto tan
largo sugiere recorte/compresión de contexto o un retriever con `k` excesivo — antes que
cambiar de modelo o de hardware. Tras el cambio, verificar con el mismo histograma.

**Ejercicio 3.** `fallback_total`, tipo **counter**, etiquetas `segmento` (3 valores) y
`version_modelo` (pocas versiones vivas): cardinalidad ≈ 3 × N_versiones, manejable. La
tasa se calcula como `fallback_total / peticiones_total` en el backend. `user_id` como
etiqueta crearía una serie por usuario (miles/millones) y reventaría el almacenamiento:
el detalle por usuario pertenece a los **logs estructurados** (consultables) y a los
atributos de la traza.

**Ejercicio 4.** En `evidence`: los eventos puntuales con contexto son logs; los
agregados numéricos (conteos, duraciones resumidas) son métricas; las secuencias
ordenadas de pasos con duración encadenada juegan el papel de traza. `limitations`
recuerda que no hay exportación OTLP real ni muestreo: todo es determinista y local.


In [ ]:
result = run_lab("observability", seed=150)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


In [ ]:
spans = {"retrieve_context": 400, "rerank": 150, "llm_call": 1700, "guardrail": 150}
total = 2400
for name, ms in spans.items():
    print(f"{name:18s} {ms:5d} ms  {100*ms/total:5.1f} %")

metrica = {
    "nombre": "fallback_total",
    "tipo": "counter",
    "etiquetas": ["segmento", "version_modelo"],
}
print(metrica)


## Reflexión

1. ¿Por qué «responde 200 OK» es un criterio de salud casi vacío para un endpoint de LLM, y qué tres señales añadirías para que «sano» signifique algo?
2. Con histogramas por span cuyo p95 conoces, ¿por qué NO puedes calcular el p95 del total, y qué señal sí te lo da?
3. ¿Qué atributos de un span `gen_ai` usarías para detectar que un cambio de prompt duplicó el costo, y qué riesgo de privacidad introduce exportar el prompt completo?
